# Mixed-Signal Audio Separation

### Recovering five hidden audio sources with Independent Component Analysis

This notebook turns a set of observed mixtures into estimated independent sources. The visual checkpoints below make the transformation easy to inspect: signal traces, frequency content, the learned unmixing matrix, and playable audio results.

> **Reproducible experiment** · fixed seed · Laplace prior · annealed stochastic gradient ascent

---

### Notebook map

1. **Prepare** the mixed signals
2. **Listen and inspect** the observations
3. **Learn** the unmixing matrix $W$
4. **Recover** and compare the separated sources

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import scipy.io.wavfile
from IPython.display import Audio, display

Fs = 11025

# Keep the notebook visually consistent and readable when exported.
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titleweight': 'bold',
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'font.size': 10,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


In [18]:
def update_W(W, x, learning_rate):
    """
    Perform a gradient ascent update on W using data element x and the provided learning rate.

    This function should return the updated W.

    Use the laplace distribiution in this problem.

    Args:
        W: The W matrix for ICA
        x: A single data element
        learning_rate: The learning rate to use

    Returns:
        The updated W
    """
    
    # *** START CODE HERE ***
    updated_W = W + learning_rate * (np.linalg.inv(W.T) - np.outer(np.sign(W.dot(x)), x.T))
    # *** END CODE HERE ***

    return updated_W

In [19]:
def unmix(X, W):
    """
    Unmix an X matrix according to W using ICA.

    Args:
        X: The data matrix
        W: The W for ICA

    Returns:
        A numpy array S containing the split data
    """

    S = np.zeros(X.shape)


    # *** START CODE HERE ***
    S = X.dot(W.T)
    # *** END CODE HERE ***

    return S


def normalize(dat):
    return 0.99 * dat / np.max(np.abs(dat))

In [20]:
def load_data():
    mix = np.loadtxt('data/mix.dat')
    return mix


def save_W(W):
    np.savetxt('output/W.txt',W)


def save_sound(audio, name):
    scipy.io.wavfile.write('output/{}.wav'.format(name), Fs, audio)

In [21]:
def unmixer(X):
    M, N = X.shape
    W = np.eye(N)

    anneal = [0.1 , 0.1, 0.1, 0.05, 0.05, 0.05, 0.02, 0.02, 0.01 , 0.01, 0.005, 0.005, 0.002, 0.002, 0.001, 0.001]
    print('Separating tracks ...')
    for lr in anneal:
        print(lr)
        rand = np.random.permutation(range(M))
        for i in rand:
            x = X[i]
            W = update_W(W, x, lr)

    return W

## 1. Load and Normalize Data
Initialize the random seed for reproducibility and load the mixed audio data.

In [22]:
# Seed the randomness of the simulation so this outputs the same thing each time
np.random.seed(0)
X = normalize(load_data())

print("Data shape:", X.shape)

Data shape: (53442, 5)


In [ ]:
# A compact overview of the observed mixtures.
time = np.arange(X.shape[0]) / Fs
preview = min(X.shape[0], Fs * 3)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), constrained_layout=True)
for track in range(X.shape[1]):
    axes[0].plot(time[:preview], X[:preview, track], linewidth=0.8, alpha=0.75, label=f'Mix {track}')
axes[0].set_title('Observed mixtures · first 3 seconds')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].legend(ncol=5, frameon=False, loc='upper right')

frequency = np.fft.rfftfreq(preview, 1 / Fs)
for track in range(X.shape[1]):
    spectrum = np.abs(np.fft.rfft(X[:preview, track]))
    axes[1].plot(frequency, spectrum, linewidth=0.9, alpha=0.8, label=f'Mix {track}')
axes[1].set_title('Frequency fingerprints')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude')
axes[1].set_xlim(0, Fs / 2)
plt.show()


## 2. Save Mixed Signals
Save the mixed sources to disk to verify the input audio.

In [23]:
if not os.path.exists('output'):
    os.makedirs('output')

print("Mixed Audio Tracks:")
for i in range(X.shape[1]):
    save_sound(X[:, i], 'mixed_{}'.format(i))
    print(f'Track {i}: output/mixed_{i}.wav')
    display(Audio(X[:, i], rate=Fs))

Mixed Audio Tracks:
Track 0: output/mixed_0.wav


Track 1: output/mixed_1.wav


Track 2: output/mixed_2.wav


Track 3: output/mixed_3.wav


Track 4: output/mixed_4.wav


## 3. Run ICA (Unmixing)
Run the `unmixer` to learn the weight matrix $W$. This process uses gradient ascent with annealing learning rates.

In [24]:
W = unmixer(X)
print("Learned W:")
print(W)
save_W(W)

Separating tracks ...
0.1
0.1
0.1
0.05
0.05
0.05
0.02
0.02
0.01
0.01
0.005
0.005
0.002
0.002
0.001
0.001
Learned W:
[[ 52.83492974  16.79598806  19.9411949  -10.19841036 -20.8977174 ]
 [ -9.9368057   -0.97879563  -4.68186342   8.0430365    1.79099473]
 [  8.31143332  -7.47699382  19.31554724  15.17460858 -14.32640472]
 [-14.66729873 -26.64481368   2.44071692  21.38223128  -8.42094492]
 [ -0.26917605  18.37373974   9.31200636   9.10275731  30.59390495]]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(W, cmap='RdBu_r', aspect='equal')
ax.set_title('Learned unmixing matrix $W$')
ax.set_xlabel('Mixture index')
ax.set_ylabel('Recovered source index')
ax.set_xticks(range(W.shape[1]))
ax.set_yticks(range(W.shape[0]))
for row in range(W.shape[0]):
    for column in range(W.shape[1]):
        ax.text(column, row, f'{W[row, column]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(image, ax=ax, label='Weight')
plt.show()


## 4. Recover Sources
Apply the learned unmixing matrix $W$ to the mixed data $X$ to get the separated signals $S$. Save the results.

In [25]:
S = normalize(unmix(X, W))
assert S.shape[1] == 5
print(f"Separated data shape: {S.shape}")

print("Separated Audio Tracks:")
for i in range(S.shape[1]):
    if os.path.exists('output/split_{}.wav'.format(i)):
        os.unlink('output/split_{}.wav'.format(i))
    save_sound(S[:, i], 'split_{}'.format(i))
    print(f'Source {i}: output/split_{i}.wav')
    display(Audio(S[:, i], rate=Fs))

Separated data shape: (53442, 5)
Separated Audio Tracks:
Source 0: output/split_0.wav


Source 1: output/split_1.wav


Source 2: output/split_2.wav


Source 3: output/split_3.wav


Source 4: output/split_4.wav


In [ ]:
# Compare the recovered signals using the same view as the input overview.
fig, axes = plt.subplots(2, 1, figsize=(14, 7), constrained_layout=True)
for source in range(S.shape[1]):
    axes[0].plot(time[:preview], S[:preview, source], linewidth=0.8, alpha=0.8, label=f'Source {source}')
axes[0].set_title('Recovered sources · first 3 seconds')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].legend(ncol=5, frameon=False, loc='upper right')

for source in range(S.shape[1]):
    spectrum = np.abs(np.fft.rfft(S[:preview, source]))
    axes[1].plot(frequency, spectrum, linewidth=0.9, alpha=0.8, label=f'Source {source}')
axes[1].set_title('Recovered frequency fingerprints')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude')
axes[1].set_xlim(0, Fs / 2)
plt.show()
